# Exploratory Data Analysis

This notebook explores the PhonePe data step by step before opportunity scoring.

The goal is to understand:

1. whether the data is complete enough for analysis,
2. how payment activity and merchant presence are distributed,
3. which states and districts show strong demand,
4. how demand relates to merchant penetration,
5. which districts deserve deeper merchant-expansion investigation.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from phonepe_analytics.metrics import add_growth_metrics, add_ratio_metrics

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data" / "processed"
CHART_DIR = ROOT / "reports" / "eda_charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

district = pd.read_csv(DATA_DIR / "district_quarter.csv")
state = pd.read_csv(DATA_DIR / "state_quarter.csv")
categories = pd.read_csv(DATA_DIR / "state_transaction_categories.csv")

district = add_ratio_metrics(district)
district = add_growth_metrics(district, ["state", "district"])

state = add_ratio_metrics(state)
state = add_growth_metrics(state, ["state"])

latest_period = district["period_id"].max()
latest_district = district[district["period_id"] == latest_period].copy()
latest_state = state[state["period_id"] == latest_period].copy()

latest_label = (
    f"{int(latest_district['year'].max())} "
    f"Q{int(latest_district['quarter'].max())}"
)

print("Latest period:", latest_label)
print("District rows:", len(latest_district))
print("States/UTs:", latest_district["state"].nunique())


## 1. Data quality and coverage

Before analysis, confirm that the latest quarter contains the core variables needed for comparison.


In [ ]:
core_columns = [
    "transaction_count",
    "transaction_amount",
    "registered_users",
    "registered_merchants",
]

missing_values = latest_district[core_columns].isna().sum()

coverage = (
    district.groupby(["year", "quarter"])
    .agg(
        district_rows=("district", "size"),
        states=("state", "nunique"),
        districts=("district", "nunique"),
    )
    .reset_index()
)

display(missing_values.rename("missing_values").to_frame())
display(coverage.tail(8))


### Insight

- The latest quarter is the main comparison period for expansion analysis.
- Missing values in the latest quarter should be investigated before ranking districts.
- Historical merchant gaps should remain missing rather than being replaced with zero, because zero would incorrectly mean that no merchants existed.


## 2. Descriptive statistics

Start with the main business variables and derived metrics.


In [ ]:
analysis_columns = [
    "transaction_count",
    "transaction_amount",
    "registered_users",
    "registered_merchants",
    "average_transaction_value",
    "merchants_per_100k_users",
    "users_per_merchant",
    "transaction_yoy",
]

descriptive_stats = (
    latest_district[analysis_columns]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9])
    .T
)

display(descriptive_stats)


### Insight

- Large differences between the mean and median indicate that district activity is unevenly distributed.
- Median and percentile comparisons are therefore more useful than relying only on averages.
- Merchant penetration should be interpreted together with user scale and transaction activity.


## 3. Univariate analysis

Examine the distributions of the variables most important to the business problem.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

sns.histplot(
    latest_district["registered_users"].dropna(),
    bins=30,
    kde=True,
    ax=axes[0, 0],
)
axes[0, 0].set_title("Registered Users")

sns.histplot(
    latest_district["registered_merchants"].dropna(),
    bins=30,
    kde=True,
    ax=axes[0, 1],
)
axes[0, 1].set_title("Registered Merchants")

sns.histplot(
    latest_district["merchants_per_100k_users"].dropna(),
    bins=30,
    kde=True,
    ax=axes[1, 0],
)
axes[1, 0].set_title("Merchants per 100K Users")

sns.histplot(
    latest_district["transaction_yoy"].dropna(),
    bins=30,
    kde=True,
    ax=axes[1, 1],
)
axes[1, 1].set_title("Transaction YoY Growth")

plt.tight_layout()
plt.savefig(CHART_DIR / "core_distributions.png", dpi=160)
plt.show()


### Insight

- User and merchant counts are strongly affected by district size, so absolute values alone are not enough.
- Merchants per 100K users gives a more comparable measure of merchant penetration.
- Transaction growth helps distinguish mature large markets from markets where demand is still expanding quickly.


## 4. State-level demand

Identify the largest state markets before moving to district-level opportunity analysis.


In [ ]:
top_states = (
    latest_state[
        [
            "state",
            "transaction_count",
            "transaction_amount",
            "registered_users",
            "registered_merchants",
        ]
    ]
    .sort_values("transaction_count", ascending=False)
    .head(15)
)

display(top_states)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=top_states,
    x="transaction_count",
    y="state",
)
plt.title(f"Top States by Transaction Volume | {latest_label}")
plt.xlabel("Transactions")
plt.ylabel("")
plt.tight_layout()
plt.savefig(CHART_DIR / "top_states_transactions.png", dpi=160)
plt.show()


### Insight

- High-volume states are commercially important, but they are not automatically the best expansion targets.
- The next step is to move to district level and compare demand with merchant penetration.


## 5. District-level demand and merchant penetration

Compare the strongest districts across three views: transaction demand, growth, and relative merchant penetration.


In [ ]:
columns = [
    "state",
    "district",
    "transaction_count",
    "registered_users",
    "registered_merchants",
    "transaction_yoy",
    "merchants_per_100k_users",
]

largest_districts = latest_district.nlargest(10, "transaction_count")[columns]

fastest_growing = (
    latest_district[latest_district["registered_users"] >= 100_000]
    .dropna(subset=["transaction_yoy"])
    .nlargest(10, "transaction_yoy")[columns]
)

lowest_penetration = (
    latest_district[
        (latest_district["registered_users"] >= 100_000)
        & (latest_district["registered_merchants"] > 0)
    ]
    .nsmallest(10, "merchants_per_100k_users")[columns]
)

print("Largest districts by transaction volume")
display(largest_districts)

print("Fastest-growing sizeable districts")
display(fastest_growing)

print("Lowest merchant penetration among sizeable districts")
display(lowest_penetration)


### Insight

- The biggest districts are not always the fastest-growing districts.
- Low merchant penetration is more meaningful when a district also has a large user base and strong transaction activity.
- Districts that repeatedly appear across these views are stronger candidates for expansion analysis.


## 6. Bivariate analysis

Explore the relationships that directly support the business question.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(
    data=latest_district,
    x="registered_users",
    y="transaction_count",
    alpha=0.5,
    ax=axes[0],
)
axes[0].set_title("Registered Users vs Transactions")

sns.scatterplot(
    data=latest_district,
    x="merchants_per_100k_users",
    y="transaction_yoy",
    size="registered_users",
    sizes=(20, 180),
    alpha=0.5,
    legend=False,
    ax=axes[1],
)
axes[1].axhline(0, linestyle="--", linewidth=1)
axes[1].set_title("Transaction Growth vs Merchant Penetration")

plt.tight_layout()
plt.savefig(CHART_DIR / "key_relationships.png", dpi=160)
plt.show()

user_transaction_corr = latest_district[
    ["registered_users", "transaction_count"]
].corr(method="spearman").iloc[0, 1]

penetration_growth_corr = latest_district[
    ["merchants_per_100k_users", "transaction_yoy"]
].corr(method="spearman").iloc[0, 1]

print("Users vs transactions Spearman correlation:", round(user_transaction_corr, 3))
print(
    "Merchant penetration vs transaction growth Spearman correlation:",
    round(penetration_growth_corr, 3),
)


### Insight

- A positive users-versus-transactions relationship confirms that market scale matters.
- The merchant-penetration-versus-growth relationship helps identify markets where demand may be growing faster than merchant registration.
- Correlation is descriptive only; it does not prove that adding merchants causes transaction growth.


## 7. Multivariate opportunity view

Combine market scale, demand growth, and merchant penetration in one simple view.


In [ ]:
opportunity_view = latest_district.dropna(
    subset=[
        "registered_users",
        "transaction_count",
        "transaction_yoy",
        "merchants_per_100k_users",
    ]
).copy()

growth_median = opportunity_view["transaction_yoy"].median()
penetration_median = opportunity_view["merchants_per_100k_users"].median()

opportunity_view["high_growth"] = (
    opportunity_view["transaction_yoy"] >= growth_median
)
opportunity_view["low_penetration"] = (
    opportunity_view["merchants_per_100k_users"] < penetration_median
)

priority_candidates = opportunity_view[
    opportunity_view["high_growth"]
    & opportunity_view["low_penetration"]
    & (opportunity_view["registered_users"] >= 100_000)
].copy()

priority_candidates = priority_candidates.sort_values(
    ["transaction_yoy", "registered_users"],
    ascending=[False, False],
)

display(
    priority_candidates[
        [
            "state",
            "district",
            "registered_users",
            "transaction_count",
            "transaction_yoy",
            "merchants_per_100k_users",
        ]
    ].head(20)
)

plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=opportunity_view,
    x="merchants_per_100k_users",
    y="transaction_yoy",
    size="registered_users",
    sizes=(20, 220),
    alpha=0.5,
    legend=False,
)
plt.axhline(growth_median, linestyle="--", linewidth=1)
plt.axvline(penetration_median, linestyle="--", linewidth=1)
plt.title("District Opportunity View")
plt.xlabel("Merchants per 100K Users")
plt.ylabel("Transaction YoY Growth")
plt.tight_layout()
plt.savefig(CHART_DIR / "district_opportunity_view.png", dpi=160)
plt.show()


### Insight

The most interesting districts are in the **high-growth / low-merchant-penetration** area.

These districts should not automatically receive expansion budget. They should move to the next stage of investigation because they combine:

- meaningful user scale,
- strong transaction demand,
- positive demand growth,
- comparatively low merchant penetration.


## 8. Transaction category context

Check what types of transactions contribute to PhonePe activity so district demand is not incorrectly treated as merchant-only demand.


In [ ]:
latest_category_period = categories["period_id"].max()

category_mix = (
    categories[categories["period_id"] == latest_category_period]
    .groupby("category", as_index=False)["transaction_count"]
    .sum()
)

category_mix["share"] = (
    category_mix["transaction_count"]
    / category_mix["transaction_count"].sum()
)

category_mix = category_mix.sort_values("share", ascending=False)

display(category_mix)

plt.figure(figsize=(9, 5))
sns.barplot(
    data=category_mix,
    x="share",
    y="category",
)
plt.title(f"Transaction Category Mix | {latest_label}")
plt.xlabel("Share of Transactions")
plt.ylabel("")
plt.tight_layout()
plt.savefig(CHART_DIR / "transaction_category_mix.png", dpi=160)
plt.show()


### Insight

- Total PhonePe transaction activity includes multiple payment categories.
- District transaction totals therefore represent overall PhonePe ecosystem demand, not purely merchant payments.
- This is an important limitation when interpreting transactions per merchant or expansion opportunity.


## 9. Final EDA findings

The EDA supports five main conclusions:

1. **Market scale matters.** Registered users and transaction activity are strongly related.
2. **Large markets are not automatically expansion markets.** High absolute transaction volume can already coexist with strong merchant presence.
3. **Merchant penetration provides the gap signal.** Merchants per 100K users makes district comparisons more meaningful.
4. **Growth helps identify momentum.** High-growth districts with relatively low merchant penetration deserve deeper investigation.
5. **The result is a prioritisation signal, not proof of business impact.** PhonePe Pulse does not contain active merchant counts, competitor coverage, merchant acquisition cost, or merchant-level revenue.

The next notebook should use these EDA findings to build and validate the merchant opportunity score.
